# Data Deduplication Lab - Getting Started

Welcome to the Data Deduplication Lab. This notebook prepares your **Cloudera AI Workbench** session for the Phase 1 exercises.

## Learning Objectives

By the end of this notebook, you will:
- Install project Python dependencies from `requirements.txt`
- Create a **local-mode** Spark session in Cloudera AI Workbench (`local[*]`)
- Read sample customer data from the project `data/` directory
- Copy that local file to HDFS with **`hdfs dfs`** CLI, then verify it
- Inspect duplicate patterns on `name` + `email`

## What this notebook does

1. **Dependencies** — install packages from `../requirements.txt`
2. **Spark setup** — create a local Spark session (local files only; no forced `fs.defaultFS`)
3. **Local read** — load `../data/redundant_data.csv` via `file://`
4. **HDFS copy** — `hdfs dfs -put` the local file under `/tmp`, then verify with CLI
5. **Duplicate check** — profile uniqueness on key columns

## Prerequisites

- Cloudera AI Workbench session with PySpark and `hdfs` / `kinit` client tools
- Project `use-case-phase-1/data/` and cluster client XML under `use-case-phase-1/hadoop-conf/`
- Kerberos user/realm configured in the Environment Settings cell (or via env vars)
- Permission to write HDFS `/tmp`
- Basic Python familiarity

## Types of Deduplication (lab overview)

### Record-Level Deduplication (Exercise 1+)
- Removes duplicate **rows/records** within a dataset
- Example: two customer rows with the same name and email

**Next notebook after this setup:** `01_Basic_Deduplication.ipynb`


## 0. Install Project Requirements

Install packages from `use-case-phase-1/requirements.txt` into this session (includes `pyspark` if it is not already on the kernel path). Re-run this cell after restarting the kernel if imports fail.


In [ ]:
from pathlib import Path
import importlib
import sys
import subprocess

REQ_FILE = Path("../requirements.txt").resolve()
assert REQ_FILE.is_file(), f"Missing requirements file: {REQ_FILE}"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])

# Ensure a fresh import after install (handles kernels that started without pyspark)
importlib.invalidate_caches()
import pyspark

print(f"✓ Installed requirements from: {REQ_FILE}")
print(f"✓ pyspark {pyspark.__version__} available")


## 0b. Environment Settings (edit per cluster / user)

Set Kerberos identity for **your** CDP environment. Client Hadoop XML lives in the project:

```text
use-case-phase-1/hadoop-conf/core-site.xml
use-case-phase-1/hadoop-conf/hdfs-site.xml
```

That folder is the usual source — **not** `/etc/hadoop/conf` (often empty in CAI).

If those files are missing, download them from:
- **Cloudera Manager** → Cluster → Actions → **Download Client Configuration**
- or **CDP** → Environment → Data Lake → Actions → **Download Client Configuration**

Unzip and place `core-site.xml` / `hdfs-site.xml` (and optional `ssl-client.xml`) into `use-case-phase-1/hadoop-conf/`.


In [ ]:
import os
from pathlib import Path

# --- Edit these defaults for your environment ---
KRB_USER = os.environ.get("KRB_USER", "your_user")
KRB_REALM = os.environ.get("KRB_REALM", "GO01-DEM.YLCU-ATMI.CLOUDERA.SITE")
KRB_DOMAIN = os.environ.get("KRB_DOMAIN", KRB_REALM.lower())
KRB_PRINCIPAL = os.environ.get("KRB_PRINCIPAL", f"{KRB_USER}@{KRB_REALM}")

def resolve_hadoop_conf() -> Path:
    """Locate project/cluster client XML. Do not export to os.environ until HDFS cells."""
    project_conf = Path("../hadoop-conf").resolve()
    candidates = []
    env_conf = os.environ.get("HADOOP_CONF_DIR")
    if env_conf:
        candidates.append(Path(env_conf))
    candidates.extend(
        [
            project_conf,
            Path("/etc/hadoop/conf"),
            Path("/etc/hadoop/conf.cloudera.hdfs"),
        ]
    )
    for conf_dir in candidates:
        if (conf_dir / "core-site.xml").is_file() and (conf_dir / "hdfs-site.xml").is_file():
            return conf_dir.resolve()
    raise FileNotFoundError(
        "No Hadoop client config found.\n"
        "Put cluster client XML in: use-case-phase-1/hadoop-conf/\n"
        "  (need at least core-site.xml and hdfs-site.xml)\n"
        "Download from Cloudera Manager / CDP Data Lake → Download Client Configuration."
    )

PROJECT_HADOOP_CONF_DIR = resolve_hadoop_conf()

# krb5.conf for kinit (safe to export; does not load Hadoop security for Spark)
KRB5_CONFIG = Path(
    os.environ.get("KRB5_CONFIG", PROJECT_HADOOP_CONF_DIR / "krb5.conf")
).resolve()
KRB5_CONFIG.parent.mkdir(parents=True, exist_ok=True)
KRB5_CONFIG.write_text(
    f"""[libdefaults]
  default_realm = {KRB_REALM}
  dns_lookup_realm = true
  dns_lookup_kdc = true
  ticket_lifetime = 24h
  renew_lifetime = 7d
  forwardable = true
  rdns = false
  udp_preference_limit = 1

[realms]
  {KRB_REALM} = {{
  }}

[domain_realm]
  .{KRB_DOMAIN} = {KRB_REALM}
  {KRB_DOMAIN} = {KRB_REALM}
"""
)

# Export identity + krb5 only. Defer HADOOP_CONF_DIR until hdfs dfs cells.
os.environ["KRB_USER"] = KRB_USER
os.environ["KRB_REALM"] = KRB_REALM
os.environ["KRB_DOMAIN"] = KRB_DOMAIN
os.environ["KRB_PRINCIPAL"] = KRB_PRINCIPAL
os.environ["KRB5_CONFIG"] = str(KRB5_CONFIG)
os.environ["PROJECT_HADOOP_CONF_DIR"] = str(PROJECT_HADOOP_CONF_DIR)
# Clear any pre-set HADOOP_CONF_DIR so local Spark does not load Kerberos core-site.xml
os.environ.pop("HADOOP_CONF_DIR", None)

print("Environment settings")
print(f"  KRB_USER                 = {KRB_USER}")
print(f"  KRB_REALM                = {KRB_REALM}")
print(f"  KRB_DOMAIN               = {KRB_DOMAIN}")
print(f"  KRB_PRINCIPAL            = {KRB_PRINCIPAL}")
print(f"  KRB5_CONFIG              = {KRB5_CONFIG}")
print(f"  PROJECT_HADOOP_CONF_DIR  = {PROJECT_HADOOP_CONF_DIR}")
print("  HADOOP_CONF_DIR          = (unset until HDFS copy cell)")
if KRB_USER == "your_user":
    print("⚠ Replace KRB_USER (or set env KRB_USER / KRB_PRINCIPAL) before kinit.")


## 1. Create Local Spark Session

This lab uses **local mode** (`local[*]`) for local `file://` reads.

**Do not** export cluster `HADOOP_CONF_DIR` before creating Spark — that loads Kerberos `core-site.xml` and fails with `Can't get Kerberos realm`. Hadoop client conf is applied later only for `hdfs dfs`.


In [ ]:
import os
from pyspark.sql import SparkSession

# HDFS destination under /tmp (used later by hdfs dfs -put)
HDFS_DIR = "/tmp"
HDFS_PATH = f"{HDFS_DIR}/redundant_data.csv"

# Local Spark must not inherit cluster Kerberos client XML
for _k in ("HADOOP_CONF_DIR", "HADOOP_HOME", "HADOOP_HDFS_HOME"):
    if _k in os.environ:
        print(f"⚠ Unsetting {_k}={os.environ[_k]} for local Spark startup")
        os.environ.pop(_k)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DeduplicationLab_GettingStarted")
    .config("spark.sql.shuffle.partitions", "8")
    # Keep local session on simple auth even if JVM picks up other defaults
    .config("spark.hadoop.hadoop.security.authentication", "simple")
    .config("spark.hadoop.hadoop.security.authorization", "false")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"HDFS path:     {HDFS_PATH}")
print("✓ Spark session created successfully")
print("⚠ Restart the kernel if a previous SparkContext failed mid-start.")


## 2. Read Local Sample Data

Load the project CSV from `use-case-phase-1/data/`. Schema: `id`, `name`, `email`, `address`.

For a larger workload later, switch the filename to `redundant_data_large.csv`.


In [ ]:
from pathlib import Path

LOCAL_DATA_DIR = Path("../data").resolve()
LOCAL_INPUT = LOCAL_DATA_DIR / "redundant_data.csv"
# Optional scale-up file:
# LOCAL_INPUT = LOCAL_DATA_DIR / "redundant_data_large.csv"

assert LOCAL_INPUT.is_file(), f"Missing local file: {LOCAL_INPUT}"

# Explicit file:// so Spark does not try to interpret the path via HDFS
LOCAL_URI = LOCAL_INPUT.resolve().as_uri()

df_local = spark.read.csv(LOCAL_URI, header=True, inferSchema=True)

print(f"✓ Loaded local file: {LOCAL_URI}")
print(f"Total records: {df_local.count():,}")
print(f"Columns: {', '.join(df_local.columns)}")
print("\nPreview:")
df_local.show(10, truncate=False)
df_local.printSchema()


## 3. Kerberos Login (`kinit`)

Uses `KRB_PRINCIPAL` from the Environment Settings cell. Replace `your_user` / realm there (or via env vars), then run this cell. You will be prompted for your password.


In [ ]:
import getpass
import os
import shutil
import subprocess

kinit_bin = shutil.which("kinit")
klist_bin = shutil.which("klist")
assert kinit_bin, "kinit not found on PATH"
assert klist_bin, "klist not found on PATH"

# From Environment Settings cell (or env)
KRB_PRINCIPAL = os.environ["KRB_PRINCIPAL"]

print(f"Using principal: {KRB_PRINCIPAL}")
print(f"KRB5_CONFIG:     {os.environ.get('KRB5_CONFIG')}")

# Skip password prompt if a ticket already exists
if subprocess.run([klist_bin, "-s"], env=os.environ).returncode == 0:
    print("✓ Kerberos ticket already present")
else:
    assert not KRB_PRINCIPAL.startswith("your_user@"), (
        "Set KRB_USER / KRB_PRINCIPAL in the Environment Settings cell "
        "(or export KRB_USER / KRB_PRINCIPAL) before kinit."
    )
    password = getpass.getpass(f"Password for {KRB_PRINCIPAL}: ")
    print(f"$ kinit {KRB_PRINCIPAL}")
    result = subprocess.run(
        [kinit_bin, KRB_PRINCIPAL],
        input=password + "\n",
        text=True,
        env=os.environ,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "kinit failed")
    print("✓ kinit succeeded")

print("$ klist")
subprocess.check_call([klist_bin], env=os.environ)


## 4. Copy Local Data to HDFS (`hdfs dfs`)

**Important:** bare `!hdfs dfs ...` fails with `core-site.xml not found` unless `HADOOP_CONF_DIR` is set in the kernel first. The cell below sets it, then runs:

```bash
hdfs dfs -mkdir -p /tmp
hdfs dfs -put -f <local-file> /tmp/redundant_data.csv
```


In [ ]:
import os
from pathlib import Path

def resolve_hadoop_conf() -> Path:
    project_conf = Path(os.environ.get("PROJECT_HADOOP_CONF_DIR", "../hadoop-conf")).resolve()
    candidates = [project_conf, Path("../hadoop-conf").resolve()]
    for conf_dir in candidates:
        if (conf_dir / "core-site.xml").is_file() and (conf_dir / "hdfs-site.xml").is_file():
            return conf_dir.resolve()
    raise FileNotFoundError(
        "Missing core-site.xml. Expected:\n"
        f"  {project_conf}/core-site.xml\n"
        "Sync use-case-phase-1/hadoop-conf/ into this CAI project."
    )

# Apply Hadoop client conf only for hdfs dfs (after Spark local session exists)
HADOOP_CONF_DIR = resolve_hadoop_conf()
KRB5_CONFIG = Path(os.environ.get("KRB5_CONFIG", HADOOP_CONF_DIR / "krb5.conf")).resolve()
os.environ["HADOOP_CONF_DIR"] = str(HADOOP_CONF_DIR)
os.environ["KRB5_CONFIG"] = str(KRB5_CONFIG)

LOCAL_FILE = str(Path(LOCAL_INPUT).resolve())
HDFS_DIR = "/tmp"
HDFS_PATH = "/tmp/redundant_data.csv"

print(f"HADOOP_CONF_DIR={os.environ['HADOOP_CONF_DIR']}")
print(f"KRB5_CONFIG={os.environ['KRB5_CONFIG']}")
print(f"local file -> {LOCAL_FILE}")
print(f"hdfs dest  -> {HDFS_PATH}")

!hdfs dfs -mkdir -p {HDFS_DIR}
!hdfs dfs -put -f {LOCAL_FILE} {HDFS_PATH}
!hdfs dfs -ls -h {HDFS_PATH}

print(f"✓ Copied to HDFS: {HDFS_PATH}")


## 5. Verify the HDFS File (CLI)

Re-check with `hdfs dfs -ls` / `-cat`. Uses the same `HADOOP_CONF_DIR` already set in `os.environ`.


In [ ]:
# Verify HDFS copy (HADOOP_CONF_DIR must already be set in os.environ)
!hdfs dfs -ls -h {HDFS_PATH}
!hdfs dfs -cat {HDFS_PATH} | head -n 6
print(f"✓ Verified HDFS file: {HDFS_PATH}")


## 6. Quick Duplicate Check (Local Data)

Profile uniqueness on `name` + `email` in the local dataset — the same keys used in Exercise 1.


In [ ]:
KEY_COLS = ["name", "email"]

total_count = df_local.count()
unique_count = df_local.select(*KEY_COLS).distinct().count()
duplicates_count = total_count - unique_count
duplicate_rate = (duplicates_count / total_count * 100) if total_count else 0

print(f"Total records: {total_count:,}")
print(f"Unique records (by name+email): {unique_count:,}")
print(f"Duplicate records: {duplicates_count:,}")
print(f"Duplicate rate: {duplicate_rate:.2f}%")
print(f"\nLocal path for Exercise 1:\n  {LOCAL_INPUT}")
print(f"HDFS path (CLI):\n  {HDFS_PATH}")


## Next Steps

Setup is complete. Exercise 1 defaults to the **local** CSV:

```text
../data/redundant_data.csv
```

HDFS copy (via `hdfs dfs -put`):

```text
/tmp/redundant_data.csv
```

1. **Exercise 1**: Basic Deduplication — `01_Basic_Deduplication.ipynb`
2. **Exercise 2**: Iceberg REST Catalog — `02_Iceberg_REST_Catalog.ipynb`

## Cleanup

**Restart the kernel** if you previously created a Spark session with `fs.defaultFS=hdfs://ns1`, then re-run from the top. Stop the Spark session when finished:


In [ ]:
spark.stop()
print("✓ Spark session stopped")
